In [0]:
import pyspark.sql.functions as F


customers = spark.read.table('workspace.erp_lakehouse.bronze_bc_customers')
items = spark.read.table('workspace.erp_lakehouse.bronze_bc_items')
orders = spark.read.table('workspace.erp_lakehouse.bronze_bc_orders')
order_lines = spark.read.table('workspace.erp_lakehouse.bronze_bc_order_lines')
# DBTITLE 1

customers.printSchema()
items.printSchema()


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

SCHEMA = "workspace.erp_lakehouse"

bc_tables = {
    "bronze_bc_customers":      "silver_bc_customers",       # bronze table -> silver table
    "bronze_bc_items":          "silver_bc_items",
    "bronze_bc_sales_orders":   "silver_bc_sales_orders",
    "bronze_bc_sales_invoices": "silver_bc_sales_invoices",
}

results = []                                                  # one summary row per table

for bronze_name, silver_name in bc_tables.items():
    df = spark.table(f"{SCHEMA}.{bronze_name}")               # read the bronze table
    bronze_n = df.count()                                     # row count before cleaning

    odata_cols = [c for c in df.columns if c.startswith("@odata")]   # find the etag column
    df = df.drop(*odata_cols)                                 # drop it, it has no business value

    df = df.withColumn("lastModifiedDateTime",
        F.expr("try_to_timestamp(lastModifiedDateTime)"))     # string -> timestamp, null if bad

    w = Window.partitionBy("id").orderBy(F.col("ingested_at").desc())   # group rows by id, newest load first
    silver_df = (df.withColumn("rn", F.row_number().over(w))  # number the rows inside each id
                   .filter("rn = 1")                          # keep only the newest row per id
                   .drop("rn"))                               # remove the helper column

    silver_n = silver_df.count()                              # row count after dedupe
    distinct_n = silver_df.select("id").distinct().count()    # must equal silver_n

    (silver_df.write.mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(f"{SCHEMA}.{silver_name}"))              # save the silver table

    results.append((bronze_name, silver_name, bronze_n, silver_n, distinct_n,
                    ", ".join(odata_cols) or "none"))         # remember what happened

summary = spark.createDataFrame(
    results,
    ["bronze_table", "silver_table", "bronze_rows", "silver_rows",
     "distinct_keys", "dropped_columns"])                     # one row per table
summary.display()                                             # check it: silver_rows = distinct_keys